#Initialization

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

#Read Bronze table

In [0]:
df = spark.table("pcat.bronze.customers")

In [0]:
df.show(10)

#Silver Transformations

##Drop Duplicates

In [0]:
duplicate_data = df.groupBy("customer_id").count().filter(F.col("count") > 1)
display(duplicate_data)

In [0]:
df = df.dropDuplicates(["customer_id"])

##Trim spaces

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, F.trim(F.col(field.name)))

##Normalization

In [0]:
df.select("city").distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}
allowed_names = ["Bengaluru", "Hyderabad", "New Delhi"]

df = (
    df
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isin(allowed_names), F.col("city"))
         .otherwise(None)
    )
)

##Fix Name casing

In [0]:
df = df.withColumn(
    "customer_name",
    F.initcap(F.col("customer_name"))
)

##Fixing City Null Values

In [0]:
df.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
#Assuming the following customers are located in the city mentioned
customer_city_mapping = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

fixed_df = spark.createDataFrame(
    [(x, y) for x, y in customer_city_mapping.items()],
    schema=["customer_id", "fixed_city"]
)

display(fixed_df)

In [0]:
df = (
    df.join(
        fixed_df,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "city",
        F.coalesce(F.col("city"), F.col("fixed_city"))
    )
    .drop("fixed_city")
)

##Convert Customer_id to string

In [0]:
df = df.withColumn(
    "customer_id",
    F.col("customer_id").cast("string")
)

##Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df = (
    df
    .withColumn(
        "customer",
        F.concat_ws(
            "-",
            F.col("customer_name"),
            F.coalesce(F.col("city"), F.lit("Unknown"))
        )
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar")) # child company
    .withColumn("channel", F.lit("Acquisition"))
)

##Check Dataframe

In [0]:
display(df.limit(10))

#Writing Silver Table

In [0]:
df.write \
  .format("delta") \
  .option("delta.enableChangeDataFeed", "true") \
  .option("mergeSchema", "true") \
  .mode("overwrite") \
  .saveAsTable("pcat.silver.customers")

##Sanity checks of silver table

In [0]:
%sql
SELECT * FROM pcat.silver.customers LIMIT 10